In [3]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
with open("data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [6]:
context_length = 8

x = enc_text[:context_length]
y = enc_text[1:context_length + 1]

print("Input tokens:", x)
print("Target tokens:   ", y)

Input tokens: [40, 367, 2885, 1464, 1807, 3619, 402, 271]
Target tokens:    [367, 2885, 1464, 1807, 3619, 402, 271, 10899]


In [11]:
import torch

print(torch.__version__)

2.8.0


In [12]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


In [13]:
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, enc_text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(enc_text, allowed_special={"<|endoftext|>"})
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_seq = token_ids[i:i + max_length]
            target_seq = token_ids[i + 1:i + max_length + 1]
            
            self.input_ids.append(torch.tensor(input_seq, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_seq, dtype=torch.long))
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [16]:
def create_dataloader(txt, batch_size = 2, max_length = 256,
                      stride = 128, shuffle = True, drop_last = True,
                      num_workers = 0):
    
    tokenizer = tiktoken.get_encoding("gpt2")
    
    dataset = GPTDataset(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                            drop_last=drop_last, num_workers=num_workers)
    
    return dataloader



In [19]:
dataloaders = create_dataloader(raw_text, batch_size=1, max_length=4, stride=1)

dataloader_iter = iter(dataloaders)
first_batch = next(dataloader_iter)
print("Input batch:", first_batch)

Input batch: [tensor([[553, 373, 465, 691]]), tensor([[ 373,  465,  691, 5402]])]


In [20]:
second_batch = next(dataloader_iter)
print("Input batch:", second_batch)

Input batch: [tensor([[ 550, 3750,  351,  262]]), tensor([[3750,  351,  262, 4269]])]


In [23]:
dataloaders = create_dataloader(raw_text, batch_size=8, max_length=4, stride=4)

dataloader_iter = iter(dataloaders)
first_batch = next(dataloader_iter)
print("Input batch:", first_batch)

Input batch: [tensor([[   13,   402,   271, 10899],
        [  407,   644,   530,  3382],
        [  379,   407,  1972,   262],
        [ 3886,   449,   659,   438],
        [ 8034,    13,   198,   198],
        [49655,  2651,    13,   366],
        [   13, 12622, 41379,   293],
        [  262,  1633,   286, 24380]]), tensor([[  402,   271, 10899,   531],
        [  644,   530,  3382,   284],
        [  407,  1972,   262, 18560],
        [  449,   659,   438,    64],
        [   13,   198,   198,   464],
        [ 2651,    13,   366,  1858],
        [12622, 41379,   293,   373],
        [ 1633,   286, 24380,   329]])]
